# EgoSchema — Frame Selection Benchmark

Runs all selectors (Uniform, Random, SigLIP-TopK) on all cached videos with
Qwen2-VL-2B as the frozen judge. Produces a comparison table with accuracy
and inference time at the end.

**Attach before running:**
- `kaggle-frame-eval` (code dataset)
- `egoschema-frames-cache` (or any other dataset cache)

**Accelerator:** GPU T4 x2 or P100. Internet: On.

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate bitsandbytes pyarrow

## 2. Paths and imports

In [ ]:
import sys, os, time, json
from pathlib import Path
import pandas as pd
import numpy as np

# ── edit these two paths if Kaggle mounts yours differently ──────────────
CODE_DIR  = "/kaggle/input/datasets/mdnomanbiswassibly/kaggle-frame-eval/kaggle-frame-eval"
CACHE_DIR = "/kaggle/input/datasets/mdnomanbiswassibly/egoschema-frames-cache"
OUTPUT_DIR = "/kaggle/working/results"
# ─────────────────────────────────────────────────────────────────────────

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

assert Path(CODE_DIR).exists(),  f"Code dir not found:  {CODE_DIR}"
assert Path(CACHE_DIR).exists(), f"Cache dir not found: {CACHE_DIR}"
print("Code dir  :", CODE_DIR)
print("Cache dir :", CACHE_DIR)
print("Contents  :", [p.name for p in Path(CACHE_DIR).iterdir()][:6])

## 3. Config

The **only cell you edit** between experiments.
To benchmark a different dataset, change `CACHE_DIR` above.
To add your own selector, add its name to `SELECTORS_TO_RUN`.

In [ ]:
# ── experiment config ────────────────────────────────────────────────────
K_FRAMES     = 8      # frames the VLM receives per question
N_EXAMPLES   = None   # None = all cached examples. 10 = quick smoke test.
SEED         = 0
VLM_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"

# All selectors to benchmark in this run.
# Add "my_algo" here once you develop your own selector.
SELECTORS_TO_RUN = ["uniform", "random", "siglip_topk"]
# ─────────────────────────────────────────────────────────────────────────

## 4. Load dataset and VLM

`VideoQACache` works with any dataset that follows the standard cache layout
(`metadata.parquet` + `frames/*.npy`). Swap `CACHE_DIR` to switch datasets —
no code changes needed.

The VLM loads once and stays in GPU memory for all selector runs.

In [ ]:
from data import VideoQACache
from vlm import Qwen2VLJudge
from frame_selectors import build_selector
from eval import run_evaluation

# Dataset — works for EgoSchema, LVB, MVBench, or any cache you build
dataset = VideoQACache(CACHE_DIR).subset(n=N_EXAMPLES, seed=SEED)
print(f"Examples to evaluate: {len(dataset)}")

# VLM — slow on first run (~3 min download), ~30 sec after that
print("\nLoading VLM...")
t0 = time.perf_counter()
judge = Qwen2VLJudge(model_id=VLM_MODEL_ID)
print(f"VLM ready in {time.perf_counter() - t0:.1f}s")

## 5. Run all selectors

Each selector runs on the **identical** set of examples so results are
directly comparable. The VLM and dataset stay in memory the whole time —
no reloading between selectors.

In [ ]:
all_metrics = {}   # selector_name -> metrics dict

for selector_name in SELECTORS_TO_RUN:
    print(f"\n{'='*55}")
    print(f"  Selector: {selector_name}")
    print(f"{'='*55}")

    # Build selector
    # SigLIP downloads ~900 MB on first use, then caches
    kwargs = {"seed": SEED} if selector_name == "random" else {}
    selector = build_selector(selector_name, **kwargs)

    run_name = f"{selector_name}_k{K_FRAMES}_n{len(dataset)}"

    t0 = time.perf_counter()
    metrics = run_evaluation(
        dataset=dataset,
        selector=selector,
        judge=judge,
        k=K_FRAMES,
        output_dir=OUTPUT_DIR,
        run_name=run_name,
    )
    wall_time = time.perf_counter() - t0

    metrics["wall_time_s"] = round(wall_time, 1)
    all_metrics[selector_name] = metrics
    print(f"  Finished in {wall_time/60:.1f} min")

print("\nAll selectors done.")

## 6. Comparison table

In [ ]:
rows = []
for name, m in all_metrics.items():
    n_total   = m['overall']['n']
    n_correct = round(m['overall']['accuracy'] * n_total)
    rows.append({
        "Selector":               name,
        "Accuracy":               f"{m['overall']['accuracy']:.4f}",
        "Correct / Total":        f"{n_correct} / {n_total}",
        "Selector Time/q (s)":    f"{m.get('mean_selector_time_s', 0):.3f}",
        "VLM Time/q (s)":         f"{m.get('mean_vlm_time_s', 0):.3f}",
        "Total Time (min)":       f"{m['wall_time_s']/60:.1f}",
    })

df_cmp = (
    pd.DataFrame(rows)
    .sort_values("Accuracy", ascending=False)
    .reset_index(drop=True)
)
df_cmp.index += 1   # rank: 1 = best

dataset_name = Path(CACHE_DIR).name
print(f"\n{'─'*68}")
print(f"  Frame Selection Benchmark")
print(f"  Dataset : {dataset_name}")
print(f"  VLM     : {VLM_MODEL_ID}")
print(f"  k={K_FRAMES}   n={len(dataset)}")
print(f"{'─'*68}")
print(df_cmp.to_string())
print(f"{'─'*68}")
print("\nSelector Time = time for selector to pick frames (per question).")
print("VLM Time      = time for Qwen2-VL to answer (per question).")

## 7. Per-question breakdown

Inspect which questions a selector got right or wrong, and which frames it chose.

In [ ]:
INSPECT_SELECTOR = "uniform"   # change to any name in SELECTORS_TO_RUN

run_name = f"{INSPECT_SELECTOR}_k{K_FRAMES}_n{len(dataset)}"
df = pd.read_csv(f"{OUTPUT_DIR}/per_example_{run_name}.csv")
df['correct'] = (df['pred_idx'] == df['correct_idx']).astype(int)

print(f"Selector : {INSPECT_SELECTOR}")
print(f"Accuracy : {df['correct'].mean():.4f}  ({df['correct'].sum()}/{len(df)})")
print()
print("=== Wrong answers ===")
cols = ['video_id', 'question', 'correct_idx', 'pred_idx',
        'selected_indices', 'selector_time_s', 'vlm_time_s']
print(df[df['correct'] == 0][cols].to_string(max_colwidth=60))

## 8. Save comparison to CSV

In [ ]:
out_path = f"{OUTPUT_DIR}/comparison_k{K_FRAMES}_n{len(dataset)}.csv"
df_cmp.to_csv(out_path, index=True)
print(f"Saved: {out_path}")
print("Download: Kaggle notebook → Output tab → download the CSV")

## Adding your own algorithm

1. Create `frame_selectors/my_algo.py` — subclass `FrameSelector`, implement `select(frames, query, k) → List[int]`.
2. Register it in `frame_selectors/__init__.py`: add `"my_algo": MyAlgo` to the `SELECTORS` dict.
3. Upload a new version of the `kaggle-frame-eval` dataset on Kaggle.
4. Add `"my_algo"` to `SELECTORS_TO_RUN` in Cell 3. Rerun from Cell 5.

**Switching datasets:** just change `CACHE_DIR` in Cell 2 to point at a
different cache (e.g. an LVB cache). `VideoQACache` reads any cache that
follows the standard schema. No other changes needed.